In [ ]:
!wget "https://raw.githubusercontent.com/SJGuy-UMN/CSCI4521/refs/heads/main/Life-Expectancy-Data-Updated.csv"

Part 1: Data Frame

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("Life-Expectancy-Data-Updated.csv")
data['HighLifeExpectancy'] = data['Life_expectancy'].apply(lambda x: x> 75.0 )
data

Part 2: Visualize

In [ ]:
features = ['BMI','Infant_deaths','Alcohol_consumption', 'Adult_mortality']
plt.figure()
sns.pairplot(data, vars=features, hue='HighLifeExpectancy')
plt.show()

In [ ]:
X = data[features].to_numpy()
y = data['HighLifeExpectancy'].to_numpy()


In [ ]:
from sklearn import neighbors
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
#split data using test_train_split before normalizing to prevent data leakage
#manual test_train split from class
def train_test_split(X, y, test_size):
  num_rows = y.size
  split = int((1-test_size)*num_rows)
  p= np.random.permutation(num_rows)
  X = X[p]
  Y = y[p]

  #training data
  X_train = X[:split] #feature_train
  Y_train = Y[:split] #label_train

  #testing data
  X_test = X[split:] #feature_test
  Y_test = Y[split:] #label_test

  return X_train, Y_train, X_test, Y_test

In [ ]:
X_train, Y_train, X_test, Y_test = train_test_split(X, y, 0.33)


In [ ]:
scalar = StandardScaler()
#normalize only the training set to prevent leakage from test set
X_train_scaled= scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

Part 3: classifiers

In [ ]:
#small k
small_knn = KNeighborsClassifier(n_neighbors=3)
small_knn.fit(X_train_scaled, Y_train)

In [ ]:
#large k
large_knn = KNeighborsClassifier(n_neighbors=10)
large_knn.fit(X_train_scaled, Y_train)

In [ ]:
def true_always(x):
  return np.ones(len(x), dtype=bool)

In [ ]:
def false_always(x):
  return np.zeros(len(x), dtype=bool)

In [ ]:
#training and testing predictions for all classifiers

y_train_small = small_knn.predict(X_train_scaled)
y_test_small = small_knn.predict(X_test_scaled)

y_train_large = large_knn.predict(X_train_scaled)
y_test_large = large_knn.predict(X_test_scaled)

y_train_true = true_always(X_train)
y_test_true = true_always(X_test)

y_train_false = false_always(X_train)
y_test_false = false_always(X_test)

Part 4: Metrics

In [ ]:
#confusion matrix
def matrix(y_true, y_pred):
  True_Positive = np.sum((y_true == 1) & (y_pred == 1))
  True_Negative = np.sum((y_true == 0) & (y_pred == 0))
  False_Positive = np.sum((y_true == 0) & (y_pred == 1))
  False_Negative = np.sum((y_true == 1) & (y_pred == 0))
  return True_Positive, True_Negative, False_Positive, False_Negative

In [ ]:
#accuracy
def accuracy(TP, TN, FP, FN):
  return (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) != 0 else 0

In [ ]:
#precision
def precision(TP, FP):
  return TP / (TP + FP) if (TP + FP) != 0 else 0

In [ ]:
#recall
def recall(TP, FN):
  return TP / (TP + FN) if (TP + FN) != 0 else 0

In [ ]:
#f1_score
def f1_score(precision, recall):
  return 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

In [ ]:
#using metrics
def metrics(y_true, y_pred):
  TP, TN, FP, FN = matrix(y_true, y_pred)
  acc = accuracy(TP, TN, FP, FN)
  prec = precision(TP, FP)
  rec = recall(TP, FN)
  f1 = f1_score(prec, rec)
  print("Accuracy: ", acc)
  print("Precision: ", prec)
  print("Recall: ", rec)
  print("F1:" ,f1)
  print("")

In [ ]:
#small knn
print('Small KNN:')
print("Train")
metrics(Y_train, y_train_small)
print("Test")
metrics(Y_test, y_test_small)

In [ ]:
#large KNN
print("Large KNN:")
print("Train")
metrics(Y_train, y_train_large)
print("Test")
metrics(Y_test, y_test_large)


In [ ]:
#always true and false metrics
print("Always True:")
print("Train")
metrics(Y_train, y_train_true)
print("Test")
metrics(Y_test, y_test_true)

print("Always False:")
print("Train")
metrics(Y_train, y_train_false)
print("Test")
metrics(Y_test, y_test_false)

Part 5: Apply the model

In [ ]:
#features used: 'BMI','Infant_deaths','Alcohol_consumption', 'Adult_mortality'
new_example_country = np.array([[27, 11, 1.3, 50]])

In [ ]:
new_example_country_scaled = scalar.transform(new_example_country)

In [ ]:
#small K seems to be the better classifier based on the metrics, but both generalize well
pred_small_k = small_knn.predict(new_example_country_scaled)
if pred_small_k:
  print("The country has a high life expectancy")
else: print("low life expectancy")

In [ ]:
pred_large_k = large_knn.predict(new_example_country_scaled)
if pred_large_k:
  print("The country has a high life expectancy")
else: print("low life expectancy")